# SC-15-Zero-Knowledge-Proofs - Preuves a Divulgation Nulle

[<< Formal Verification](../03-Foundry-Testing/SC-14-Formal-Verification.ipynb) | [Homomorphic Encryption >>](SC-16-Homomorphic-Encryption.ipynb)

---

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre les **preuves a divulgation nulle** (Zero-Knowledge Proofs)
2. Implementer le **protocole de Schnorr** from scratch
3. Comprendre la **transformation de Fiat-Shamir** (interactif -> non-interactif)
4. Explorer les **Sigma protocols** et le protocole de Chaum-Pedersen

### Prerequis

- Python 3.10+ avec `hashlib` (stdlib)
- `pycryptodome` pour la generation de grands nombres premiers
- Notions de base en arithmetique modulaire

### Duree estimee : 60 minutes

---

## 1. Introduction aux preuves a divulgation nulle

### Qu'est-ce qu'une preuve a divulgation nulle ?

Une **Zero-Knowledge Proof (ZKP)** est un protocole cryptographique qui permet
a un **prouveur** (Prover) de convaincre un **verificateur** (Verifier) qu'une
affirmation est vraie, **sans reveler aucune information au-dela de la veracite
de l'affirmation elle-meme**.

### L'analogie de la caverne d'Ali Baba

Imaginez une caverne en forme d'anneau avec une porte magique au fond.
Peggy (le prouveur) connait le mot de passe de la porte. Victor (le verificateur) veut s'en assurer
sans apprendre le mot de passe.

```
     Entree
       |
      / \
     A   B
      \ /
    [Porte]
```

1. Peggy entre dans la caverne par A ou B (au hasard, Victor ne voit pas)
2. Victor crie "sors par A" ou "sors par B" (au hasard)
3. Si Peggy connait le mot de passe, elle peut toujours sortir du bon cote
4. Si elle ne le connait pas, elle a 50% de chance d'echouer a chaque tour
5. Apres N tours, la probabilite de tricher est de 2^(-N)

### Trois proprietes fondamentales

| Propriete | Description | Formellement |
|-----------|-------------|-------------|
| **Completude** | Si l'affirmation est vraie, le verificateur sera convaincu | Pr[Accept | vrai] = 1 |
| **Solidite** | Si l'affirmation est fausse, le prouveur ne peut pas tricher | Pr[Accept | faux] <= 2^(-N) |
| **Zero-knowledge** | Le verificateur n'apprend rien de plus que la veracite | Simulation indistinguable |

In [1]:
import random

def cave_simulation(knows_password: bool, num_rounds: int = 20) -> bool:
    """Simuler le protocole de la caverne d'Ali Baba.
    Retourne True si le verificateur est convaincu apres tous les rounds.
    """
    for round_num in range(num_rounds):
        # Peggy choisit un cote au hasard
        peggy_side = random.choice(["A", "B"])
        # Victor demande un cote au hasard
        victor_request = random.choice(["A", "B"])

        if peggy_side == victor_request:
            # Pas besoin de la porte, elle est deja du bon cote
            continue
        elif knows_password:
            # Peggy traverse la porte magique
            continue
        else:
            # Peggy ne peut pas traverser -> echec
            return False
    return True

# Simuler un prouveur honnete (connait le mot de passe)
print("ANALOGIE DE LA CAVERNE D'ALI BABA")
print("=" * 60)

honnete_succes = sum(cave_simulation(True, 20) for _ in range(1000))
print(f"Prouveur honnete  (1000 essais, 20 rounds) : {honnete_succes}/1000 reussis")

# Simuler un prouveur malhonnete (ne connait PAS le mot de passe)
malhonnete_succes = sum(cave_simulation(False, 20) for _ in range(1000))
print(f"Prouveur fraudeur (1000 essais, 20 rounds) : {malhonnete_succes}/1000 reussis")
print()
print(f"Probabilite theorique de fraude sur 20 rounds : 2^(-20) = {2**-20:.10f}")
print(f"-> Environ 1 chance sur {2**20:,} de tromper le verificateur")

ANALOGIE DE LA CAVERNE D'ALI BABA
Prouveur honnete  (1000 essais, 20 rounds) : 1000/1000 reussis
Prouveur fraudeur (1000 essais, 20 rounds) : 0/1000 reussis

Probabilite theorique de fraude sur 20 rounds : 2^(-20) = 0.0000009537
-> Environ 1 chance sur 1,048,576 de tromper le verificateur


### Interpretation

Le prouveur honnete reussit **toujours** (completude), tandis que le fraudeur
echoue presque certainement apres 20 rounds (solidite).

La propriete zero-knowledge vient du fait que Victor ne peut pas distinguer
"Peggy connait le mot de passe" de "Peggy a eu de la chance sur le cote".
Victor n'apprend que "oui, elle sait" ou "non, elle ne sait pas".

Ce principe se generalise a des problemes mathematiques utiles en cryptographie,
comme le **logarithme discret** que nous explorons dans la section suivante.

---

## 2. ZKP du logarithme discret (version simplifiee)

Le probleme du **logarithme discret** est fondamental en cryptographie :

> Etant donne un groupe cyclique de generateur g, et une valeur y = g^x mod p,
> retrouver x est (en pratique) impossible pour de grands nombres premiers.

Un prouveur qui connait x (le "secret") peut prouver qu'il le connait
sans jamais le reveler, grace au protocole suivant :

### Protocole interactif

1. **Engagement (Commit)** : Le prouveur choisit r aleatoire, calcule R = g^r mod p, envoie R
2. **Defi (Challenge)** : Le verificateur envoie un challenge aleatoire c
3. **Reponse (Response)** : Le prouveur calcule s = r + c*x mod q, envoie s
4. **Verification** : Le verificateur verifie que g^s == R * y^c (mod p)

### Pourquoi ca marche ?

```
g^s = g^(r + c*x) = g^r * g^(c*x) = R * (g^x)^c = R * y^c  (mod p)
```

Le prouveur ne revele jamais x directement. La valeur s = r + c*x est
"masquee" par le nonce aleatoire r, rendant impossible de deduire x.

In [2]:
from Crypto.Util.number import getPrime, getRandomRange
import hashlib

def generate_dlog_params(bits=256):
    """Generer les parametres pour un groupe de Schnorr.
    On genere p premier tel que p = 2q + 1 (premier sur),
    et g generateur du sous-groupe d'ordre q.
    """
    # Pour la pedagogie, on utilise une methode simplifiee
    # En production, on utiliserait des parametres standardises (RFC 7919)
    while True:
        q = getPrime(bits)
        p = 2 * q + 1
        # Verifier que p est premier (safe prime)
        if pow(2, p - 1, p) == 1:  # Test de Fermat rapide
            break

    # Trouver un generateur du sous-groupe d'ordre q
    while True:
        h = getRandomRange(2, p - 1)
        g = pow(h, 2, p)  # g = h^2 mod p est d'ordre q (car p = 2q+1)
        if g > 1:
            break

    return p, q, g

# Generer les parametres du groupe
print("GENERATION DES PARAMETRES DU GROUPE")
print("=" * 60)
p, q, g = generate_dlog_params(bits=128)  # 128 bits pour la demo (rapide)
print(f"p (premier sur) : {str(p)[:40]}... ({p.bit_length()} bits)")
print(f"q (ordre)       : {str(q)[:40]}... ({q.bit_length()} bits)")
print(f"g (generateur)  : {str(g)[:40]}... ({g.bit_length()} bits)")
print()

# Le secret du prouveur
x = getRandomRange(1, q)  # Cle privee
y = pow(g, x, p)          # Cle publique
print(f"Cle privee x    : {str(x)[:40]}... (SECRET)")
print(f"Cle publique y  : {str(y)[:40]}... (g^x mod p)")

GENERATION DES PARAMETRES DU GROUPE


p (premier sur) : 517263357537302337117739288257192732347... (129 bits)
q (ordre)       : 258631678768651168558869644128596366173... (128 bits)
g (generateur)  : 474978373304843484574352515986480774670... (129 bits)

Cle privee x    : 36435626521820701096336190484713880656... (SECRET)
Cle publique y  : 370636417868986398703602946901174957022... (g^x mod p)


### Execution du protocole interactif

Avec ces parametres en place, executons le protocole en 4 etapes.
A chaque round, le prouveur genere un nonce frais r, ce qui masque
completement le secret x dans la reponse s = r + c*x.

In [ ]:
# Protocole interactif du logarithme discret
from Crypto.Util.number import getRandomRange

def zkp_dlog_interactive(g, y, x, p, q):
    """Executer une ronde du protocole ZKP interactif.
    g, y, p, q : parametres publics
    x : secret du prouveur
    Retourne (succes, details)
    """
    # Etape 1 : Engagement (Prover)
    r = getRandomRange(1, q)  # Nonce aleatoire
    R = pow(g, r, p)          # Commitment

    # Etape 2 : Challenge (Verifier)
    c = getRandomRange(1, q)  # Challenge aleatoire

    # Etape 3 : Response (Prover)
    s = (r + c * x) % q       # Reponse

    # Etape 4 : Verification (Verifier)
    lhs = pow(g, s, p)        # g^s mod p
    rhs = (R * pow(y, c, p)) % p  # R * y^c mod p

    return lhs == rhs, {"R": R, "c": c, "s": s, "lhs": lhs, "rhs": rhs}


# Executer le protocole plusieurs fois
print("PROTOCOLE ZKP DU LOGARITHME DISCRET (INTERACTIF)")
print("=" * 60)

num_rounds = 10
all_valid = True

for i in range(num_rounds):
    valid, details = zkp_dlog_interactive(g, y, x, p, q)
    status = "OK" if valid else "ECHEC"
    print(f"  Round {i+1:2d} : g^s == R*y^c ? {status}")
    if not valid:
        all_valid = False

resultat = "CONVAINCU" if all_valid else "ECHEC"
print()
print(f"Resultat apres {num_rounds} rounds : {resultat}")
print(f"Probabilite de fraude : 2^(-{num_rounds}) = {2**-num_rounds:.2e}")
print()
print("Points cles :")
print("  - Le secret x n'a JAMAIS ete transmis au verificateur")
print("  - Chaque round utilise un nonce r different (aleatoire)")
print("  - s = r + c*x est masque par r, impossible de deduire x")

### Et si le prouveur triche ?

Pour comprendre la solidite du protocole, voyons ce qui se passe
quand un prouveur ne connait **pas** le secret x et tente de generer
une reponse valide.

In [4]:
# Demonstration : un prouveur frauduleux echoue
print("TENTATIVE DE FRAUDE (prouveur ne connait PAS x)")
print("=" * 60)

def zkp_dlog_cheat(g, y, p, q):
    """Tentative de fraude : le prouveur ne connait pas x."""
    # Le fraudeur choisit un r au hasard
    r = getRandomRange(1, q)
    R = pow(g, r, p)

    # Challenge du verificateur
    c = getRandomRange(1, q)

    # Le fraudeur ne peut pas calculer s = r + c*x car il ne connait pas x
    # Il essaie un s au hasard
    s_fake = getRandomRange(1, q)

    lhs = pow(g, s_fake, p)
    rhs = (R * pow(y, c, p)) % p

    return lhs == rhs


num_attempts = 1000
frauds_reussies = sum(zkp_dlog_cheat(g, y, p, q) for _ in range(num_attempts))
print(f"Tentatives de fraude : {num_attempts}")
print(f"Fraudes reussies     : {frauds_reussies}")
print(f"-> La probabilite de deviner s correctement est 1/q ~ 2^(-{q.bit_length()})")
print("-> Un seul round suffit pour une securite de 128 bits avec de grands parametres")

TENTATIVE DE FRAUDE (prouveur ne connait PAS x)


Tentatives de fraude : 1000
Fraudes reussies     : 0
-> La probabilite de deviner s correctement est 1/q ~ 2^(-128)
-> Un seul round suffit pour une securite de 128 bits avec de grands parametres


### Interpretation

| Scenario | Resultat | Explication |
|----------|----------|-------------|
| Prouveur honnete (connait x) | Toujours accepte | s = r + c*x verifie l'equation |
| Prouveur fraudeur (ne connait pas x) | Toujours rejete | Impossible de calculer s sans x |

La securite repose sur le **probleme du logarithme discret** : meme en observant
y = g^x, R = g^r, et s = r + c*x, il est calculatoirement infaisable de retrouver x.

Limitation de la version interactive : elle necessite un echange en temps reel
entre le prouveur et le verificateur. La section suivante resout ce probleme.

---

## 3. Protocole de Schnorr et transformation de Fiat-Shamir

Le **protocole de Schnorr** (1989) est la version formalisee du protocole
de la section 2, avec une amelioration cruciale : la **transformation de Fiat-Shamir**
qui le rend **non-interactif**.

### Idee de Fiat-Shamir (1986)

Au lieu d'attendre un challenge aleatoire du verificateur, le prouveur le
**genere lui-meme** a partir du hash de l'engagement :

```
c = SHA-256(g || y || R)
```

Cela fonctionne car le hash est imprevisible et lie a l'engagement R.
Le prouveur ne peut pas choisir c a l'avance (il faudrait inverser SHA-256).

### Consequence fondamentale

La preuve non-interactive (R, s) est en fait une **signature numerique** !
Les signatures de Schnorr sont exactement des ZKP non-interactives du logarithme discret.
C'est la base de nombreux schemas modernes (BIP-340/Taproot dans Bitcoin).

In [5]:
from Crypto.Util.number import getPrime, getRandomRange
from Crypto.Hash import SHA256

class SchnorrZKP:
    """Implementation du protocole de Schnorr (interactif et non-interactif)."""

    def __init__(self, bits=128):
        """Generer les parametres du groupe."""
        while True:
            self.q = getPrime(bits)
            self.p = 2 * self.q + 1
            if pow(2, self.p - 1, self.p) == 1:
                break
        while True:
            h = getRandomRange(2, self.p - 1)
            self.g = pow(h, 2, self.p)
            if self.g > 1:
                break

    def keygen(self):
        """Generer une paire cle privee / cle publique."""
        x = getRandomRange(1, self.q)  # Cle privee
        y = pow(self.g, x, self.p)     # Cle publique
        return x, y

    # --- Version interactive ---

    def prove_interactive_step1(self, x):
        """Prouveur : etape 1 - engagement."""
        r = getRandomRange(1, self.q)
        R = pow(self.g, r, self.p)
        return r, R

    def verify_challenge(self):
        """Verificateur : generer un challenge aleatoire."""
        return getRandomRange(1, self.q)

    def prove_interactive_step3(self, r, c, x):
        """Prouveur : etape 3 - reponse."""
        s = (r + c * x) % self.q
        return s

    def verify_interactive(self, y, R, c, s):
        """Verificateur : verification finale."""
        lhs = pow(self.g, s, self.p)
        rhs = (R * pow(y, c, self.p)) % self.p
        return lhs == rhs

    # --- Version non-interactive (Fiat-Shamir) ---

    def _fiat_shamir_challenge(self, y, R, message=b""):
        """Calculer le challenge via hash (transformation de Fiat-Shamir)."""
        h = SHA256.new()
        h.update(self.g.to_bytes(256, 'big'))
        h.update(y.to_bytes(256, 'big'))
        h.update(R.to_bytes(256, 'big'))
        h.update(message)
        # Convertir le hash en entier modulo q
        return int.from_bytes(h.digest(), 'big') % self.q

    def prove_non_interactive(self, x, y, message=b""):
        """Preuve non-interactive (Fiat-Shamir)."""
        r = getRandomRange(1, self.q)
        R = pow(self.g, r, self.p)
        c = self._fiat_shamir_challenge(y, R, message)
        s = (r + c * x) % self.q
        return R, s  # La preuve = (R, s)

    def verify_non_interactive(self, y, R, s, message=b""):
        """Verification non-interactive."""
        c = self._fiat_shamir_challenge(y, R, message)
        lhs = pow(self.g, s, self.p)
        rhs = (R * pow(y, c, self.p)) % self.p
        return lhs == rhs


# Instancier le protocole
schnorr = SchnorrZKP(bits=128)
x, y = schnorr.keygen()

print("PROTOCOLE DE SCHNORR")
print("=" * 60)
print(f"Parametres : p={str(schnorr.p)[:30]}... ({schnorr.p.bit_length()} bits)")
print(f"Cle publique y = g^x mod p")
print()

PROTOCOLE DE SCHNORR
Parametres : p=554452293842134481016447573724... (129 bits)
Cle publique y = g^x mod p



### Utilisation : interactif vs non-interactif

La classe `SchnorrZKP` ci-dessus implemente les deux variantes.
Comparons les en execution pour voir la difference pratique :
l'interactif necessite 3 messages, le non-interactif un seul.

In [ ]:
# Comparaison interactive vs non-interactive
print("COMPARAISON : INTERACTIF vs NON-INTERACTIF (Fiat-Shamir)")
print("=" * 60)

# 1. Version interactive
print("\n--- Version INTERACTIVE ---")
print("  Etape 1 (Prouveur -> Verificateur) : envoyer R")
r, R = schnorr.prove_interactive_step1(x)
print(f"    R = g^r = {str(R)[:30]}...")

print("  Etape 2 (Verificateur -> Prouveur) : envoyer c")
c = schnorr.verify_challenge()
print(f"    c = {str(c)[:30]}...")

print("  Etape 3 (Prouveur -> Verificateur) : envoyer s")
s = schnorr.prove_interactive_step3(r, c, x)
print(f"    s = r + c*x mod q = {str(s)[:30]}...")

valid_inter = schnorr.verify_interactive(y, R, c, s)
inter_status = "OUI" if valid_inter else "NON"
print(f"  Verification : g^s == R*y^c ? {inter_status}")
print(f"  -> Necessite 3 messages (2 allers-retours)")

# 2. Version non-interactive (Fiat-Shamir)
print("\n--- Version NON-INTERACTIVE (Fiat-Shamir) ---")
message = b"Je prouve que je connais x"
R_ni, s_ni = schnorr.prove_non_interactive(x, y, message)
print(f"  Preuve (R, s) calculee localement")
print(f"    R = {str(R_ni)[:30]}...")
print(f"    s = {str(s_ni)[:30]}...")
print(f"    c = SHA256(g || y || R || message) (calcule automatiquement)")

valid_ni = schnorr.verify_non_interactive(y, R_ni, s_ni, message)
ni_status = "VALIDE" if valid_ni else "INVALIDE"
print(f"  Verification : {ni_status}")
print(f"  -> Un seul message, pas besoin d'interaction !")

# 3. Tentative de falsification du message
print("\n--- Falsification du message ---")
fake_message = b"Message modifie"
valid_fake = schnorr.verify_non_interactive(y, R_ni, s_ni, fake_message)
fake_status = "VALIDE" if valid_fake else "INVALIDE"
print(f"  Verification avec message modifie : {fake_status}")
print(f"  -> La preuve est liee au message (c'est une signature !)")

### ZKP = Signature numerique

Un resultat remarquable : la preuve non-interactive (R, s) est exactement
une **signature de Schnorr** sur un message. Signer, c'est prouver la
connaissance de la cle privee sans la reveler.

In [7]:
# La preuve non-interactive de Schnorr EST une signature
print("SCHNORR : ZKP = SIGNATURE")
print("=" * 60)
print()
print("Schema de signature de Schnorr :")
print("  Signer(x, message)   -> (R, s) = preuve ZKP non-interactive")
print("  Verifier(y, message, R, s) -> vrai/faux")
print()

# Signer plusieurs messages
messages = [
    b"Transferer 10 ETH a 0xBob",
    b"Approuver le contrat 0xDefi",
    b"Voter OUI pour la proposition 42",
]

print("Signatures de Schnorr sur differents messages :")
for msg in messages:
    R_sig, s_sig = schnorr.prove_non_interactive(x, y, msg)
    valid = schnorr.verify_non_interactive(y, R_sig, s_sig, msg)
    print(f"  Message  : {msg.decode()}")
    print(f"  Sig (R)  : {str(R_sig)[:30]}...")
    print(f"  Valide   : {valid}")
    print()

print("Points cles :")
print("  - Chaque signature a un R different (nonce aleatoire)")
print("  - La verification ne necessite que la cle publique y")
print("  - Bitcoin utilise les signatures Schnorr depuis Taproot (BIP-340)")
print("  - Avantage sur ECDSA : linearite -> aggregation de signatures possible")

SCHNORR : ZKP = SIGNATURE

Schema de signature de Schnorr :
  Signer(x, message)   -> (R, s) = preuve ZKP non-interactive
  Verifier(y, message, R, s) -> vrai/faux

Signatures de Schnorr sur differents messages :
  Message  : Transferer 10 ETH a 0xBob
  Sig (R)  : 312753813960364170480981450300...
  Valide   : True

  Message  : Approuver le contrat 0xDefi
  Sig (R)  : 512170747138649403014659170622...
  Valide   : True

  Message  : Voter OUI pour la proposition 42
  Sig (R)  : 451255729865399607970689808787...
  Valide   : True

Points cles :
  - Chaque signature a un R different (nonce aleatoire)
  - La verification ne necessite que la cle publique y
  - Bitcoin utilise les signatures Schnorr depuis Taproot (BIP-340)
  - Avantage sur ECDSA : linearite -> aggregation de signatures possible


### Interpretation

La transformation de Fiat-Shamir est un resultat fondamental en cryptographie :

| Aspect | Interactif | Non-interactif (Fiat-Shamir) |
|--------|-----------|------------------------------|
| Messages | 3 (aller-retour) | 1 (prouveur -> verificateur) |
| Challenge | Aleatoire (verificateur) | Hash deterministe (SHA-256) |
| Utilisation | Identification temps reel | Signatures, blockchain |
| Securite | Random Oracle Model | CRS Model (+ hash) |

**Consequence pratique** : une preuve ZKP non-interactive peut etre verifiee
par n'importe qui, a n'importe quel moment, sans interaction avec le prouveur.
C'est exactement ce dont on a besoin pour les transactions blockchain.

---

## 4. Sigma Protocols

Les protocoles de Schnorr et du logarithme discret sont des cas particuliers
d'une famille plus large : les **Sigma protocols** (protocoles en 3 etapes).

### Structure generale

Tout Sigma protocol suit le schema :

```
Prouveur                          Verificateur
  |                                    |
  |--- (1) Engagement (a) ----------->|
  |                                    |
  |<-- (2) Challenge (c) -------------|  (forme Sigma: la lettre grecque)
  |                                    |
  |--- (3) Reponse (z) ------------->|
  |                                    |
  |          Verification : V(a, c, z) = vrai/faux
```

La forme du diagramme d'echange ressemble a la lettre grecque Sigma, d'ou le nom.

### Protocole de Chaum-Pedersen

Le protocole de **Chaum-Pedersen** (1992) prouve que deux logarithmes discrets
sont **egaux** sans reveler leur valeur :

> Prouver que log_g(y1) == log_h(y2) sans reveler l'exposant commun x
>
> Ou : y1 = g^x mod p et y2 = h^x mod p

Ceci est utile pour les **votes electroniques**, les **transferts confidentiels**,
et les **preuves d'egalite** de Diffie-Hellman.

In [8]:
from Crypto.Util.number import getRandomRange
from Crypto.Hash import SHA256

class ChaumPedersen:
    """Protocole de Chaum-Pedersen : prouver log_g(y1) == log_h(y2).
    Utilise les memes parametres de groupe que Schnorr.
    """

    def __init__(self, p, q, g):
        self.p = p
        self.q = q
        self.g = g
        # Generer un second generateur h (independant de g)
        while True:
            k = getRandomRange(2, q)
            self.h = pow(g, k, p)
            if self.h > 1 and self.h != g:
                break

    def setup(self, x):
        """Calculer y1 = g^x et y2 = h^x."""
        y1 = pow(self.g, x, self.p)
        y2 = pow(self.h, x, self.p)
        return y1, y2

    def prove(self, x, y1, y2):
        """Generer la preuve (non-interactive via Fiat-Shamir)."""
        # Engagement : r aleatoire, R1 = g^r, R2 = h^r
        r = getRandomRange(1, self.q)
        R1 = pow(self.g, r, self.p)
        R2 = pow(self.h, r, self.p)

        # Challenge (Fiat-Shamir)
        hasher = SHA256.new()
        for val in [self.g, self.h, y1, y2, R1, R2]:
            hasher.update(val.to_bytes(256, 'big'))
        c = int.from_bytes(hasher.digest(), 'big') % self.q

        # Reponse
        s = (r + c * x) % self.q

        return R1, R2, s

    def verify(self, y1, y2, R1, R2, s):
        """Verifier la preuve."""
        # Recalculer le challenge
        hasher = SHA256.new()
        for val in [self.g, self.h, y1, y2, R1, R2]:
            hasher.update(val.to_bytes(256, 'big'))
        c = int.from_bytes(hasher.digest(), 'big') % self.q

        # Verifier les deux equations simultanement
        check1 = pow(self.g, s, self.p) == (R1 * pow(y1, c, self.p)) % self.p
        check2 = pow(self.h, s, self.p) == (R2 * pow(y2, c, self.p)) % self.p

        return check1 and check2


# Utiliser les parametres de Schnorr
cp = ChaumPedersen(schnorr.p, schnorr.q, schnorr.g)

# Le secret x et les deux "cles publiques"
secret_x = getRandomRange(1, schnorr.q)
y1, y2 = cp.setup(secret_x)

print("PROTOCOLE DE CHAUM-PEDERSEN")
print("=" * 60)
print(f"Secret x : {str(secret_x)[:30]}... (cache)")
print(f"y1 = g^x : {str(y1)[:30]}...")
print(f"y2 = h^x : {str(y2)[:30]}...")
print(f"\nObjectif : prouver que log_g(y1) == log_h(y2) sans reveler x")
print()

PROTOCOLE DE CHAUM-PEDERSEN
Secret x : 567744681017151405944173209174... (cache)
y1 = g^x : 499875821683500302214224391905...
y2 = h^x : 183318859273529833377947277955...

Objectif : prouver que log_g(y1) == log_h(y2) sans reveler x



### Verification et test de solidite

Verifions que la preuve passe pour des logarithmes egaux, et
echoue quand les logarithmes sont differents (le prouveur tente de
prouver une affirmation fausse).

In [ ]:
# Generer et verifier la preuve de Chaum-Pedersen
R1, R2, s = cp.prove(secret_x, y1, y2)
valid = cp.verify(y1, y2, R1, R2, s)

print("VERIFICATION CHAUM-PEDERSEN")
print("=" * 60)
print(f"Preuve (R1, R2, s) :")
print(f"  R1 = {str(R1)[:30]}...")
print(f"  R2 = {str(R2)[:30]}...")
print(f"  s  = {str(s)[:30]}...")
valid_status = "VALIDE" if valid else "INVALIDE"
print(f"\nVerification : g^s == R1*y1^c  ET  h^s == R2*y2^c")
print(f"Resultat : {valid_status}")
print()

# Test avec des exposants differents (la preuve doit echouer)
print("--- Test avec exposants differents ---")
x_diff = getRandomRange(1, schnorr.q)
y2_fake = pow(cp.h, x_diff, cp.p)  # y2 = h^(x_diff) avec x_diff != x

# Tenter de prouver avec le mauvais secret
R1_f, R2_f, s_f = cp.prove(secret_x, y1, y2_fake)
valid_fake = cp.verify(y1, y2_fake, R1_f, R2_f, s_f)
fake_status = "VALIDE" if valid_fake else "INVALIDE"
print(f"y1 = g^x, y2 = h^x_diff (x != x_diff)")
print(f"La preuve avec le secret x pour y2 = h^x_diff : {fake_status}")
print()
print("-> La preuve ne fonctionne que si les deux logarithmes sont identiques")
print()
print("Applications :")
print("  - Vote electronique : prouver qu'un bulletin chiffre est bien forme")
print("  - Transferts confidentiels : prouver qu'un montant est conserve")
print("  - Echange de cles : prouver la coherence de Diffie-Hellman")

### Notion de OR-proofs

Les Sigma protocols se **composent** naturellement :

- **AND-proof** : prouver que l'on connait x1 ET x2 (facile : executer deux preuves)
- **OR-proof** : prouver que l'on connait x1 OU x2 sans reveler lequel

L'OR-proof est plus subtile. L'idee est de **simuler** la branche que l'on
ne connait pas (grace a la propriete zero-knowledge) et de calculer la vraie
preuve pour l'autre branche, en liant les deux challenges par c = c1 + c2.

```
Prouveur connait x1 (mais pas x2) :
  1. Simuler la preuve pour x2 : choisir c2, s2, calculer R2 en arriere
  2. Calculer R1 honnetement avec r1 aleatoire
  3. Recevoir le challenge global c
  4. Calculer c1 = c - c2, puis s1 = r1 + c1*x1
  -> Le verificateur ne peut pas distinguer quelle branche est simulee
```

Applications : systemes de **vote anonyme** (prouver que mon bulletin
est 0 ou 1 sans dire lequel), **ring signatures** (Monero).

---

## 5. zk-SNARKs : vue d'ensemble

Les protocoles vus jusqu'ici prouvent des relations algebriques simples
(logarithme discret, egalite de logarithmes). Les **zk-SNARKs** generalisent
les ZKP a des calculs arbitraires.

### Qu'est-ce qu'un zk-SNARK ?

**zk-SNARK** = Zero-Knowledge Succinct Non-interactive ARgument of Knowledge

| Composante | Signification |
|-----------|---------------|
| **Zero-Knowledge** | Ne revele rien au-dela de la veracite |
| **Succinct** | Preuve courte (~200 octets), verification rapide (~ms) |
| **Non-interactive** | Pas d'echange entre prouveur et verificateur |
| **ARgument** | Securite computationnelle (pas information-theorique) |
| **of Knowledge** | Le prouveur connait reellement le temoin (extractabilite) |

### Pipeline de construction

```
Programme -> Circuit Arithmetique -> R1CS -> QAP -> Preuve
```

1. **Circuit arithmetique** : le calcul est decompose en additions et multiplications
   sur un corps fini (comme une puce electronique logique)
2. **R1CS** (Rank-1 Constraint System) : chaque porte du circuit devient une contrainte
   de la forme `(A.s) * (B.s) = (C.s)` ou s est le vecteur temoin
3. **QAP** (Quadratic Arithmetic Program) : les contraintes R1CS sont encodees comme
   des polynomes, et la verification se reduit a une identite polynomiale
4. **Preuve** : le prouveur calcule la preuve via des pairings sur courbes elliptiques

### Applications blockchain

| Projet | Utilisation des zk-SNARKs |
|--------|-------------------------|
| **Zcash** | Transactions privees (montants et adresses caches) |
| **zk-Rollups** (zkSync, StarkNet) | Scalabilite L2 : prouver N transactions en 1 preuve |
| **Tornado Cash** | Mixer : prouver l'appartenance a un ensemble sans reveler laquelle |
| **Filecoin** | Proof-of-Replication (stocker des donnees) |
| **Mina Protocol** | Blockchain de taille fixe (22 Ko) grace aux preuves recursives |

In [ ]:
# Demonstration simplifiee : circuit arithmetique et R1CS
# On veut prouver que l'on connait x tel que x^3 + x + 5 == 35 (reponse : x=3)
# sans reveler x

print("CIRCUIT ARITHMETIQUE SIMPLIFIE")
print("=" * 60)
print("Equation a prouver : x^3 + x + 5 == 35")
print("(Le prouveur connait x=3, le verificateur ne doit pas l'apprendre)")
print()

# Decomposition en circuit (chaque ligne = une porte multiplication)
# Variables : x, sym1, y, sym2, out
# Porte 1 : sym1 = x * x        (x^2)
# Porte 2 : y    = sym1 * x      (x^3)
# Porte 3 : sym2 = y + x         (x^3 + x)  -- addition, pas une porte R1CS
# Porte 4 : out  = sym2 + 5      (x^3 + x + 5)
# Contrainte : out == 35

# Le "temoin" complet (toutes les variables intermediaires)
x_secret = 3
sym1 = x_secret * x_secret       # 9
y_val = sym1 * x_secret          # 27
sym2 = y_val + x_secret          # 30
out = sym2 + 5                   # 35

witness = {
    "1": 1,          # Constante
    "x": x_secret,
    "sym1": sym1,    # x^2
    "y": y_val,      # x^3
    "sym2": sym2,    # x^3 + x
    "out": out,      # x^3 + x + 5
}

print("Temoin (witness) - toutes les variables intermediaires :")
for name, val in witness.items():
    print(f"  {name:5s} = {val}")
print()

# Contraintes R1CS : (A.s) * (B.s) = (C.s)
# Format : chaque contrainte est un triplet (A, B, C) de vecteurs
# Les vecteurs indexent : [1, x, sym1, y, sym2, out]
constraints = [
    # Porte 1 : sym1 = x * x -> (0,1,0,0,0,0)*(0,1,0,0,0,0) = (0,0,1,0,0,0)
    {"A": {"x": 1}, "B": {"x": 1}, "C": {"sym1": 1}},
    # Porte 2 : y = sym1 * x -> (0,0,1,0,0,0)*(0,1,0,0,0,0) = (0,0,0,1,0,0)
    {"A": {"sym1": 1}, "B": {"x": 1}, "C": {"y": 1}},
    # Porte 3+4 combinee : (y + x + 5) * 1 = out
    {"A": {"y": 1, "x": 1, "1": 5}, "B": {"1": 1}, "C": {"out": 1}},
]

print("Contraintes R1CS (Rank-1 Constraint System) :")
print("  Chaque contrainte : (A . witness) * (B . witness) = (C . witness)")
print()

all_valid = True
for i, con in enumerate(constraints):
    # Calculer les produits scalaires
    a_val = sum(coeff * witness[var] for var, coeff in con["A"].items())
    b_val = sum(coeff * witness[var] for var, coeff in con["B"].items())
    c_val = sum(coeff * witness[var] for var, coeff in con["C"].items())

    valid = (a_val * b_val) == c_val
    all_valid = all_valid and valid

    a_str = " + ".join(f"{c}*{v}" for v, c in con["A"].items())
    b_str = " + ".join(f"{c}*{v}" for v, c in con["B"].items())
    c_str = " + ".join(f"{c}*{v}" for v, c in con["C"].items())
    status = "OK" if valid else "ECHEC"
    print(f"  Contrainte {i+1}: ({a_str}) * ({b_str}) = ({c_str})")
    print(f"    => {a_val} * {b_val} = {c_val} ? {status}")

print(f"\nToutes les contraintes satisfaites : {all_valid}")
print(f"Le prouveur connait x tel que x^3 + x + 5 == {out}")
print()
print("En vrai zk-SNARK :")
print("  1. Le temoin serait encode comme polynomes sur un corps fini")
print("  2. La preuve serait ~200 octets (2 elements de courbe elliptique)")
print("  3. La verification prendrait ~5ms (3 pairings)")
print("  4. Le verificateur n'apprendrait pas x")

### Comparaison des systemes de preuves

| Systeme | Taille preuve | Temps verif. | Setup de confiance | Utilise par |
|---------|--------------|-------------|-------------------|------------|
| **Schnorr** | ~64 octets | ~1ms | Non | Bitcoin (Taproot) |
| **Groth16** (zk-SNARK) | ~200 octets | ~5ms | Oui (trusted setup) | Zcash, Tornado Cash |
| **PLONK** (zk-SNARK) | ~400 octets | ~10ms | Universel | zkSync, Aztec |
| **zk-STARK** | ~50 Ko | ~50ms | Non | StarkNet, Polygon Miden |
| **Bulletproofs** | ~700 octets | ~50ms | Non | Monero |

**Compromis fondamental** : taille de preuve vs. hypotheses de confiance.
Les zk-STARKs sont plus gros mais n'ont pas besoin de "trusted setup",
ce qui les rend plus transparents pour les systemes decentralises.

> **Note** : les implementations completes de zk-SNARKs et zk-STARKs depassent
> le cadre de ce notebook. Elles reposent sur des pairings de courbes elliptiques
> (BN254, BLS12-381) et de l'algebre polynomiale avancee.

---

## 6. Exercice : ZKP de preimage de hash

### Objectif

Implementer un protocole ZKP qui prouve que l'on connait la **preimage**
d'un hash SHA-256 sans la reveler.

Concretement : etant donne `h = SHA256(secret)`, prouver que l'on connait
`secret` sans le communiquer au verificateur.

### Approche

On utilise un schema de **commit-and-reveal** avec defi aleatoire :

1. Le prouveur genere un nonce `r` aleatoire
2. Il calcule `commitment = SHA256(secret || r)`
3. Le verificateur envoie un challenge `c` (0 ou 1)
4. Si c=0 : le prouveur revele `r` et le verificateur verifie que
   `SHA256(secret || r)` correspond au commitment (sans apprendre secret)
   -- en fait le verificateur ne peut pas verifier sans secret !

**Approche corrigee** (schema de Guillou-Quisquater simplifie) :

On utilise plutot une approche basee sur le logarithme discret.
Le prouveur encode son secret comme exposant et utilise le protocole de Schnorr.

### A implementer

Completez la classe `HashPreimageZKP` ci-dessous.

**Indice** : convertissez le hash de la preimage en un element du groupe
et utilisez le protocole de Schnorr pour prouver la connaissance de cet element.

In [11]:
import hashlib
from Crypto.Util.number import getRandomRange
from Crypto.Hash import SHA256


class HashPreimageZKP:
    """ZKP prouvant la connaissance d'une preimage de hash.

    Strategie : convertir le secret en un scalaire dans le groupe de Schnorr,
    puis utiliser le protocole de Schnorr pour prouver la connaissance.

    Le verificateur connait :
      - Le hash public h_pub = SHA256(secret)
      - Les parametres du groupe (p, q, g)
      - La valeur y = g^(int(h_pub) mod q) mod p

    Le prouveur connait :
      - Le secret (dont le hash donne h_pub)
    """

    def __init__(self, p, q, g):
        """Initialiser avec les parametres du groupe."""
        self.p = p
        self.q = q
        self.g = g

    def register(self, secret: bytes):
        """Enregistrer un secret et publier (h_pub, y).

        TODO:
        1. Calculer h_pub = SHA256(secret).hexdigest()
        2. Convertir h_pub en entier x_derived = int(h_pub, 16) % self.q
        3. Calculer y = pow(self.g, x_derived, self.p)
        4. Retourner (h_pub, y, x_derived)
           h_pub et y sont publics, x_derived est le secret du prouveur
        """
        pass  # TODO: Implementez register()


### Indices

Si vous etes bloque :

1. Pour `register` : utilisez `hashlib.sha256(secret).hexdigest()` puis `int(..., 16) % self.q`
2. Pour `prove` : c'est exactement le protocole de Schnorr non-interactif avec x_derived comme secret
3. Pour `verify` : recalculez le challenge de la meme maniere et verifiez l'equation
4. Pour le hash dans `prove`/`verify`, concatenez les representations en bytes des grands entiers :
   `SHA256.new(g.to_bytes(256, 'big') + y.to_bytes(256, 'big') + R.to_bytes(256, 'big'))`

---

## 7. Resume

### Recapitulatif des protocoles

| Protocole | Prouve que... | Interactif ? | Complexite |
|-----------|---------------|-------------|------------|
| **Caverne Ali Baba** | "Je connais le mot de passe" | Oui (N rounds) | O(N) |
| **Schnorr** | "Je connais x tel que g^x = y" | Les deux | O(1) |
| **Chaum-Pedersen** | "log_g(y1) = log_h(y2)" | Les deux | O(1) |
| **OR-proof** | "Je connais x1 ou x2" | Les deux | O(1) |
| **zk-SNARK** | "Je connais un temoin w pour C(w)=1" | Non | O(1) verification |

### Concepts cles

| Concept | Description |
|---------|-------------|
| **Completude** | Le prouveur honnete convainc toujours |
| **Solidite** | Le fraudeur echoue avec forte probabilite |
| **Zero-knowledge** | Le verificateur n'apprend rien de plus |
| **Fiat-Shamir** | Transformer interactif en non-interactif via hash |
| **Sigma protocol** | Framework general : engagement -> challenge -> reponse |
| **R1CS** | Representation des calculs en contraintes rang-1 |

### Points cles

- Les ZKP permettent de prouver sans reveler : un changement de paradigme en cryptographie
- Le protocole de Schnorr est la brique de base, utilisee dans Bitcoin (Taproot)
- La transformation de Fiat-Shamir convertit toute preuve interactive en signature
- Les zk-SNARKs generalisent les ZKP a des calculs arbitraires (zk-rollups, confidentialite)
- Les applications blockchain sont en pleine explosion : scalabilite L2, votes, identite

---

**Notebook suivant** : [SC-16-Homomorphic-Encryption](SC-16-Homomorphic-Encryption.ipynb) - Chiffrement homomorphe

---

[<< Formal Verification](../03-Foundry-Testing/SC-14-Formal-Verification.ipynb) | [Homomorphic Encryption >>](SC-16-Homomorphic-Encryption.ipynb)